# Experiment 10: All-26-Layers Branch 2 with Tensor Caching, Gradient Descent Core Finding & 99% Compression Sweep

**Target Model**: `google/gemma-3-1b-it` (All 26 Transformer Decoder Layers: `model.layers[0...25].mlp.gate_proj`)  
**Evaluation Task**: GLUE MNLI Validation Set (`validation_matched`)  

### Key Pipeline Principles:
1. **Single-Pass 26-Layer Profiling**: Simultaneous forward hooks capture empirical activation trajectories during uncompressed baseline inference ($W_{\text{orig}}$).
2. **Multi-Modal Non-Mutually Exclusive Grouping**: For each layer, extracts top-$M$ ($M=3$) modes per coordinate. Groups coordinates by minimal 1D absolute difference similarity to cluster anchors ($C_i \cap C_j \neq \emptyset$).
3. **Tensor Assembly & Memory Caching**: Pre-assembles all 26 non-mutually exclusive tensors $\mathcal{T}_l \in \mathbb{R}^{6 \times 400 \times 1152}$ into `cached_layer_tensors = [T_0, ..., T_25]`.
4. **Layer-Wise Superweight Quarantine**: Outlier coordinates ($|x| > 3.0$ or top 1% variance) preserved in pristine FP32.
5. **Gradient Descent Core Finding (Adam Optimization)**:
   Initializes Tucker factors via SVD/HOSVD, then optimizes the core tensor $\mathcal{S}$ and factor matrices using PyTorch Adam gradient descent to minimize reconstruction error:
   $$\min_{\mathcal{S}, A, B, C} \|\mathcal{T}_l - \mathcal{S} \times_1 A \times_2 B \times_3 C\|_F^2$$
6. **Overlap-Aware Mean Weight Composition**:
   Reconstructs weights by averaging contributions across active cluster memberships for shared coordinates:
   $$\hat{W}_l[c, :] = \frac{1}{|\mathcal{K}_c|} \sum_{k \in \mathcal{K}_c} \hat{W}_{l, k}[\text{local\_idx}(c, k), :]$$
7. **Compression Sweep Starting from ~99%**:
   Evaluates compression tiers starting from ultra-high parameter cut (98.8% cut, ranks $[2, 20, 20]$), down through aggressive and moderate tiers on the full model.

In [ ]:
# =====================================================================
# STEP 1: Environment Setup, Time Logging & Library Imports
# =====================================================================
import os
import sys
import time
from pathlib import Path
import math
import json
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import torch
import tensorly as tl
from tensorly.decomposition import tucker
from tensorly.tucker_tensor import tucker_to_tensor
from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics import accuracy_score
from IPython import get_ipython

# Set TensorLy PyTorch backend
tl.set_backend("pytorch")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Minimal Time Logging
GLOBAL_NOTEBOOK_START_TIME = time.time()
NOTEBOOK_TIMINGS = []
_current_cell_start = None

ip = get_ipython()
if ip is not None:
    def _pre_cell_hook(info):
        global _current_cell_start
        _current_cell_start = time.time()

    def _post_cell_hook(result):
        global _current_cell_start
        if _current_cell_start is not None:
            elapsed = time.time() - _current_cell_start
            cumulative = time.time() - GLOBAL_NOTEBOOK_START_TIME
            cell_id = result.execution_count or len(NOTEBOOK_TIMINGS) + 1

            timing_entry = {
                "cell_id": cell_id,
                "time": round(elapsed, 3),
                "cummulative_time": round(cumulative, 3),
            }
            NOTEBOOK_TIMINGS.append(timing_entry)

            print(f"time: {elapsed:.2f}s")
            print(f"cummulative_time: {cumulative:.2f}s")

    ip.events.register("pre_run_cell", _pre_cell_hook)
    ip.events.register("post_run_cell", _post_cell_hook)

# Neural Decomp framework imports
try:
    from neural_decomp import ModelManagementInterface, DeviceMapOptions
    from neural_decomp.decomposition import decompose_svd, truncate_svd, decompose_tucker
    from neural_decomp.utils import get_device_info, save_json_metrics
    print("Loaded neural_decomp library.")
except ImportError:
    from utility import ModelManagementInterface, DeviceMapOptions
    from utility.decomposition import decompose_svd, truncate_svd, decompose_tucker
    from utility.utils import get_device_info, save_json_metrics
    print("Loaded utility library.")

device_info = get_device_info()
print(f"Device: {device_info['device_name']} | CUDA Available: {device_info['cuda_available']}")

In [ ]:
# =====================================================================
# STEP 2: Initialize Model & Tokenizer Across All 26 Layers
# =====================================================================
model_id = "google/gemma-3-1b-it"

mmi = ModelManagementInterface(
    model_id=model_id,
    precision=torch.float32,
    device_map=DeviceMapOptions.AUTO,
)
model = mmi.get_model()
tokenizer = mmi.get_tokenizer()

NUM_LAYERS = len(model.model.layers)
print(f"Loaded model with {NUM_LAYERS} transformer decoder layers.")

# Save pristine unprocessed weights for all 26 gate_proj layers for rollbacks
W_gate_orig_all = {
    l: model.model.layers[l].mlp.gate_proj.weight.data.clone()
    for l in range(NUM_LAYERS)
}
print(f"Stored pristine baseline weights for all {NUM_LAYERS} gate_proj modules.")

In [ ]:
# =====================================================================
# STEP 3: Load GLUE MNLI Validation Benchmark
# =====================================================================
ds = load_dataset("nyu-mll/glue", "mnli")["validation_matched"]

label_names = ["entailment", "neutral", "contradiction"]
label_token_ids = [tokenizer.encode(" " + name, add_special_tokens=False)[0] for name in label_names]

# Calibration & benchmark evaluation subset
EVAL_SAMPLE_COUNT = 150
eval_data = ds.select(range(EVAL_SAMPLE_COUNT))

print(f"Loaded GLUE MNLI: {len(ds):,} total samples | Active Evaluation Subset: {len(eval_data):,} samples")

## Step 1: Single-Pass Model-Wide Activation Profiling Across All 26 Layers

We attach forward hooks across all 26 decoder layers during uncompressed baseline inference.
We capture the empirical activation trajectories and compute the pristine baseline accuracy on GLUE MNLI.

In [ ]:
# =====================================================================
# STEP 4: Initial Baseline Evaluation & Simultaneous 26-Layer Hooking
# =====================================================================
layer_trajectories = {l: [] for l in range(NUM_LAYERS)}
current_acts = {}

def make_act_hook(layer_idx):
    def hook_fn(module, input_tensor, output_tensor):
        act = output_tensor[0] if isinstance(output_tensor, tuple) else output_tensor
        current_acts[layer_idx] = act.detach().cpu()
    return hook_fn

hooks = [
    model.model.layers[l].mlp.act_fn.register_forward_hook(make_act_hook(l))
    for l in range(NUM_LAYERS)
]

predictions = []
ground_truth = []

model.eval()
with torch.no_grad():
    for sample in tqdm(eval_data, desc="Baseline Eval & 26-Layer Profiling"):
        prompt = (
            f"<start_of_turn>user\n"
            f"Premise: {sample['premise']}\n"
            f"Hypothesis: {sample['hypothesis']}\n"
            f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
            f"Answer with one word\n"
            f"<start_of_turn>model\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model(**inputs)

        for l in range(NUM_LAYERS):
            if l in current_acts and current_acts[l] is not None:
                pooled = current_acts[l].squeeze(0).mean(dim=0).numpy()
                layer_trajectories[l].append(pooled)

        next_token_logits = outputs.logits[0, -1, :]
        candidate_logits = next_token_logits[label_token_ids]
        pred_label = torch.argmax(candidate_logits).item()

        predictions.append(pred_label)
        ground_truth.append(sample["label"])

for h in hooks:
    h.remove()

acts_matrices = {l: np.stack(layer_trajectories[l]) for l in range(NUM_LAYERS)}

baseline_accuracy = accuracy_score(ground_truth, predictions)
print(f"\nUncompressed Baseline Accuracy: {baseline_accuracy * 100:.2f}%")
print(f"Profiled activation trajectories across all {NUM_LAYERS} layers.")

## Step 2: Pre-Assemble & Cache All 26 Non-Mutually Exclusive Tensors

For each layer $l$:
1. Outlier filter isolates superweights ($|x| > 3.0$ or top 1% variance) in FP32.
2. Extracts top-$M$ ($M=3$) frequent activation modes per coordinate.
3. Groups coordinates into $K=6$ clusters based on 1D absolute difference similarity to cluster anchors ($C_i \cap C_j \neq \emptyset$).
4. Assembles non-mutually exclusive 3D tensors $\mathcal{T}_l \in \mathbb{R}^{6 \times 400 \times 1152}$ and caches them in `cached_layer_tensors = [T_0, ..., T_25]`.

In [ ]:
# =====================================================================
# STEP 5: Assemble & Cache Non-Mutually Exclusive Tensors
# =====================================================================
NUM_CLUSTERS = 6
COORDS_PER_CLUSTER = 400
TOP_K_MODES = 3

cached_layer_tensors = []
layer_metadata = []
layer_multiplicities = []

for l in range(NUM_LAYERS):
    acts_l = acts_matrices[l]
    W_gate_l = W_gate_orig_all[l]
    
    # 1. Outlier Filter
    max_mags = np.max(np.abs(acts_l), axis=0)
    variances = np.var(acts_l, axis=0)
    super_mask = (max_mags > 3.0) | (variances >= np.quantile(variances, 0.99))
    super_indices = np.where(super_mask)[0]
    
    inactive_mask = (np.mean(np.abs(acts_l) < 0.05, axis=0) > 0.90) & (~super_mask)
    normal_active_indices = np.where((~super_mask) & (~inactive_mask))[0]
    
    # 2. Extract Multi-Modal Top-K Frequent Modes (Branch 2)
    rounded_acts = np.round(acts_l[:, normal_active_indices], decimals=1)
    coord_top_modes = {}
    all_modes = []
    
    for idx in range(len(normal_active_indices)):
        vals, counts = np.unique(rounded_acts[:, idx], return_counts=True)
        sorted_order = np.argsort(-counts)
        top_modes = vals[sorted_order[:TOP_K_MODES]].tolist()
        coord_top_modes[idx] = top_modes
        all_modes.extend(top_modes)
        
    all_modes = np.array(all_modes)
    cluster_anchors = np.quantile(all_modes, np.linspace(0.05, 0.95, NUM_CLUSTERS))
    
    # 3. Non-Mutually Exclusive Grouping
    num_normal = len(normal_active_indices)
    dist_matrix = np.zeros((NUM_CLUSTERS, num_normal))
    for k, anchor in enumerate(cluster_anchors):
        for idx in range(num_normal):
            modes = coord_top_modes[idx]
            dist_matrix[k, idx] = min(abs(v - anchor) for v in modes)
            
    clusters_dict = {}
    cluster_slices = []
    for k in range(NUM_CLUSTERS):
        ranked_local = np.argsort(dist_matrix[k])[:COORDS_PER_CLUSTER]
        c_coords = normal_active_indices[ranked_local]
        clusters_dict[k] = c_coords
        cluster_slices.append(W_gate_l[c_coords, :].float().cpu())
        
    all_assigned = np.concatenate([clusters_dict[k] for k in range(NUM_CLUSTERS)])
    unique_assigned = np.unique(all_assigned)
    multiplicity = len(all_assigned) / len(unique_assigned)
    layer_multiplicities.append(multiplicity)
    
    T_l = torch.stack(cluster_slices, dim=0)
    cached_layer_tensors.append(T_l)
    
    layer_metadata.append({
        "layer": l,
        "super_indices": super_indices,
        "clusters_dict": clusters_dict,
        "unique_coords": len(unique_assigned),
        "multiplicity": round(multiplicity, 3),
        "d_in": W_gate_l.shape[1],
        "num_coords": W_gate_l.shape[0],
    })

print(f"Successfully assembled and cached all {len(cached_layer_tensors)} non-mutually exclusive tensors.")
print(f"Mean Cluster Multiplicity across all layers: {np.mean(layer_multiplicities):.2f} clusters/coord")

## Step 3: Gradient Descent Core Finding & Factor Optimization

We define `optimize_tucker_gd()` to refine the core tensor $\mathcal{S}$ and factor matrices via Adam gradient descent from SVD initialization.

In [ ]:
# =====================================================================
# STEP 6: Define Gradient Descent Core Finding Function
# =====================================================================
def optimize_tucker_gd(T, ranks, num_steps=30, lr=1e-3, device="cpu"):
    """
    Performs Tucker decomposition with SVD initialization followed by
    gradient descent refinement of the core tensor and factor matrices.
    """
    # 1. HOSVD Initialization
    core_init, factors_init = tucker(T, rank=ranks, init='svd')
    
    # 2. Convert to trainable PyTorch parameters
    core_param = torch.nn.Parameter(core_init.clone().to(device))
    factors_param = [torch.nn.Parameter(f.clone().to(device)) for f in factors_init]
    
    optimizer = torch.optim.Adam([core_param] + factors_param, lr=lr)
    T_target = T.to(device)
    
    with torch.no_grad():
        T_recon_init = tucker_to_tensor((core_param, factors_param))
        init_err = (torch.norm(T_target - T_recon_init) / torch.norm(T_target)).item()
        
    for step in range(num_steps):
        optimizer.zero_grad()
        T_recon = tucker_to_tensor((core_param, factors_param))
        loss = torch.norm(T_target - T_recon) ** 2
        loss.backward()
        optimizer.step()
        
    with torch.no_grad():
        T_recon_final = tucker_to_tensor((core_param, factors_param)).cpu()
        final_err = (torch.norm(T.cpu() - T_recon_final) / torch.norm(T.cpu())).item()
        
    return core_param.detach().cpu(), [f.detach().cpu() for f in factors_param], T_recon_final, init_err, final_err

# Quick demo on Layer 0 cached tensor
demo_ranks = [4, 180, 600]
core_opt, factors_opt, T_demo_recon, svd_err, gd_err = optimize_tucker_gd(cached_layer_tensors[0], demo_ranks, num_steps=30)
print(f"Layer 0 GD Core Optimization Demo (Branch 2, Ranks {demo_ranks}):")
print(f"  SVD Init Recon Error: {svd_err * 100:.2f}%")
print(f"  GD Refined Recon Error: {gd_err * 100:.2f}% (Loss minimized via Adam)")

## Step 4: Multi-Tier Compression Sweep Starting from ~99% with Overlap Composition

We evaluate compression tiers starting from ultra-high parameter cut down to balanced ranks:
- **Tier 1: Ultra-High (~99% cut / 86x on active tensor)**: Ranks $[2, 20, 20]$
- **Tier 2: Aggressive (~89% cut / 8.9x on active tensor)**: Ranks $[3, 80, 200]$
- **Tier 2.5: Target (~77% cut / 4.35x on active tensor)**: Ranks $[4, 120, 360]$
- **Tier 3: Moderate (~57% cut / 2.3x on active tensor)**: Ranks $[4, 180, 600]$

Reconstruction applies **mean aggregation across active memberships** for overlapping coordinates.

In [ ]:
# =====================================================================
# STEP 7: Run GD-Tucker Decomposition with Overlap Composition
# =====================================================================
compression_tiers = [
    {"label": "Tier 1 (Ultra ~99% Cut)", "ranks": [2, 20, 20]},
    {"label": "Tier 2 (Aggressive ~89% Cut)", "ranks": [3, 80, 200]},
    {"label": "Tier 2.5 (Target ~77% Cut)",   "ranks": [4, 120, 360]},
    {"label": "Tier 3 (Moderate ~57% Cut)",   "ranks": [4, 180, 600]},
]

tier_results = []

for tier_cfg in compression_tiers:
    tier_label = tier_cfg["label"]
    ranks = tier_cfg["ranks"]
    print(f"\n{'='*75}")
    print(f"Executing: {tier_label} with Ranks {ranks} (Branch 2) across all 26 layers...")
    print(f"{'='*75}")
    
    total_orig_gate = 0
    total_comp_gate = 0
    layer_errors = []
    
    for l in range(NUM_LAYERS):
        T_l = cached_layer_tensors[l]
        meta = layer_metadata[l]
        
        # Optimize core & factors via Adam GD
        core_opt, factors_opt, T_l_recon, svd_err, gd_err = optimize_tucker_gd(
            T_l, ranks, num_steps=25, lr=1e-3
        )
        layer_errors.append(gd_err)
        
        # Parameter accounting
        core_params = core_opt.numel()
        factor_params = sum(f.numel() for f in factors_opt)
        comp_active = core_params + factor_params
        orig_active = T_l.numel()
        
        full_orig = W_gate_orig_all[l].numel()
        full_comp = full_orig - orig_active + comp_active
        total_orig_gate += full_orig
        total_comp_gate += full_comp
        
        # Overlap-Aware Weight Composition
        W_reconstructed = W_gate_orig_all[l].clone().float().cpu()
        weight_accum = torch.zeros_like(W_gate_orig_all[l], dtype=torch.float32, device="cpu")
        membership_counts = torch.zeros(W_gate_orig_all[l].shape[0], dtype=torch.float32, device="cpu")
        
        for k in range(NUM_CLUSTERS):
            coords_t = torch.tensor(meta["clusters_dict"][k], dtype=torch.long)
            weight_accum.index_add_(0, coords_t, T_l_recon[k].float().cpu())
            membership_counts.index_add_(0, coords_t, torch.ones(len(coords_t), dtype=torch.float32))
            
        comp_mask = membership_counts > 0
        W_reconstructed[comp_mask] = weight_accum[comp_mask] / membership_counts[comp_mask].unsqueeze(1)
        W_reconstructed[meta["super_indices"], :] = W_gate_orig_all[l][meta["super_indices"], :].float().cpu()
        
        target_mod = model.model.layers[l].mlp.gate_proj
        target_mod.weight.data = W_reconstructed.to(device=model.device, dtype=target_mod.weight.dtype)
        
    global_cut_pct = (1.0 - total_comp_gate / total_orig_gate) * 100.0
    global_ratio = total_orig_gate / total_comp_gate
    mean_err = np.mean(layer_errors) * 100.0
    
    print(f"Global Gate Params: {total_comp_gate:,} vs {total_orig_gate:,} ({global_ratio:.2f}x, {global_cut_pct:.2f}% cut)")
    print(f"Mean Layer Recon Error: {mean_err:.2f}%")
    
    # Evaluate full model on GLUE MNLI
    print(f"Evaluating {tier_label} on GLUE MNLI...")
    tier_preds, tier_gts = [], []
    model.eval()
    with torch.no_grad():
        for sample in tqdm(eval_data, desc=f"Eval {tier_label}"):
            prompt = (
                f"<start_of_turn>user\n"
                f"Premise: {sample['premise']}\n"
                f"Hypothesis: {sample['hypothesis']}\n"
                f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
                f"Answer with one word\n"
                f"<start_of_turn>model\n"
            )
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            outputs = model(**inputs)
            next_token_logits = outputs.logits[0, -1, :]
            pred_label = torch.argmax(next_token_logits[label_token_ids]).item()
            tier_preds.append(pred_label)
            tier_gts.append(sample["label"])
            
    tier_acc = accuracy_score(tier_gts, tier_preds)
    delta = tier_acc - baseline_accuracy
    print(f"  Accuracy: {tier_acc * 100:.2f}% (Δ vs Baseline: {delta * 100:+.2f}%)")
    
    tier_results.append({
        "Tier": tier_label,
        "Ranks": str(ranks),
        "Total Params": total_comp_gate,
        "Ratio": f"{global_ratio:.2f}x",
        "Cut %": f"{global_cut_pct:.2f}%",
        "Mean Err %": f"{mean_err:.2f}%",
        "Accuracy %": f"{tier_acc * 100:.2f}%",
        "Delta": f"{delta * 100:+.2f}%",
    })

# Restore clean baseline
for l in range(NUM_LAYERS):
    model.model.layers[l].mlp.gate_proj.weight.data = W_gate_orig_all[l].clone()
print("\nAll 26 layer baseline weights restored.")

## Step 5: Summary Table, Comparative Synthesis & Artifact Export

We tabulate the multi-tier compression sweep results for Branch 2, comparing accuracy and parameter savings against Branch 1.

In [ ]:
# =====================================================================
# STEP 8: Summary Table & Artifact Export
# =====================================================================
# Attempt to load Branch 1 GD sweep results for direct comparison
branch1_gd_path = Path("artifacts/09_all_layers_gd_branch1_results.json")
b1_accs = {}
if branch1_gd_path.exists():
    try:
        with open(branch1_gd_path, "r") as f:
            b1_data = json.load(f)
            for row in b1_data.get("results", []):
                b1_accs[row["Tier"]] = row.get("Accuracy %", "-")
    except Exception:
        pass

summary_rows = [
    {
        "Tier": "Baseline (Uncompressed)",
        "Ranks": "Full",
        "Total Params": 26 * 6912 * 1152,
        "Ratio": "1.00x",
        "Cut %": "0.00%",
        "Mean Err %": "0.00%",
        "Accuracy %": f"{baseline_accuracy * 100:.2f}%",
        "Delta": "+0.00%",
    }
] + tier_results

print("=" * 115)
print(f"{'Tier':<30} | {'Ranks':<15} | {'Params':<10} | {'Ratio':<6} | {'Cut %':<7} | {'Err %':<8} | {'Branch 2 Acc':<13} | {'Delta':<7}")
print("=" * 115)
for r in summary_rows:
    print(f"{r['Tier']:<30} | {r['Ranks']:<15} | {r['Total Params']:<10} | {r['Ratio']:<6} | {r['Cut %']:<7} | {r['Mean Err %']:<8} | {r['Accuracy %']:<13} | {r['Delta']:<7}")
print("=" * 115)

# Global Notebook Timing
total_notebook_runtime = time.time() - GLOBAL_NOTEBOOK_START_TIME
print(f"cummulative_time: {total_notebook_runtime:.2f}s")

# Save export artifact
artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)
output_path = artifact_dir / "10_all_layers_gd_branch2_results.json"

export_data = {
    "experiment": "10_all_layers_gd_core_branch2",
    "model_id": model_id,
    "num_layers": NUM_LAYERS,
    "eval_samples": len(eval_data),
    "mean_multiplicity": round(np.mean(layer_multiplicities), 3),
    "baseline_accuracy": baseline_accuracy,
    "results": summary_rows,
    "timing_summary": {
        "cummulative_time_sec": round(total_notebook_runtime, 3),
        "cell_timings": NOTEBOOK_TIMINGS,
    }
}

save_json_metrics(export_data, output_path)
print(f"\nExperiment 10 results saved to: {output_path}")